# Semana 29 — Propensity Score Matching e Instrumental Variables
**Fase 6: Causal Inference e Experimentation** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/notebooks/semana_29_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-29-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/docs/Semana-29-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

## Parte A — Propensity Score Matching

### Passo 1 — O cenário
Um app de fitness lança um "Personal Trainer de IA". Queremos saber se usá-lo reduz o colesterol. O uso é voluntário: pessoas mais jovens, que já se exercitam e com mais consciência de saúde adotam mais. Efeito verdadeiro: **−8 mg/dL**.

In [ ]:
EFEITO_PT = -8.0
def gerar_fitness(n=6000, seed=0):
    r = np.random.default_rng(seed)
    idade = r.normal(42, 11, n).clip(18, 75); exerc = r.gamma(2, 1.5, n).clip(0, 12); consc = r.normal(0, 1, n)   # horas/semana; consciência de saúde
    logit = -0.6 - 0.04 * (idade - 42) + 0.35 * (exerc - 3) + 0.9 * consc
    usa = r.binomial(1, 1 / (1 + np.exp(-logit)))
    colesterol = 205 + 0.6 * (idade - 42) - 2.5 * (exerc - 3) - 6 * consc + EFEITO_PT * usa + r.normal(0, 12, n)
    return pd.DataFrame({"idade": idade, "exerc": exerc, "consc": consc, "usa": usa, "colesterol": colesterol})

df = gerar_fitness(); print(df.groupby("usa").mean().round(2))

### Passo 2 — A comparação ingênua

In [ ]:
ingenuo = df[df.usa == 1]["colesterol"].mean() - df[df.usa == 0]["colesterol"].mean()
print(f"diferença ingênua: {ingenuo:.1f} mg/dL | verdade: {EFEITO_PT}")

O app parece duas vezes melhor do que é. De onde vem o excesso?

### Passo 3 — Imbalance: standardized mean difference (SMD)
`SMD = (média_tratados − média_controles) / desvio-padrão agrupado`. Valores acima de 0,1 em módulo indicam imbalance.

In [ ]:
COV = ["idade", "exerc", "consc"]
def smd(d, col, peso=None):
    t, c = d[d.usa == 1], d[d.usa == 0]
    return (t[col].mean() - c[col].mean()) / np.sqrt((t[col].var() + c[col].var()) / 2)
print({c: round(smd(df, c), 2) for c in COV})

### Passo 4 — Estimar o propensity score
Regressão logística de `usa` nas três covariáveis. Guarde a probabilidade prevista em `df["ps"]`.

In [ ]:
df["ps"] = LogisticRegression(max_iter=1000).fit(df[COV], df["usa"]).predict_proba(df[COV])[:, 1]
plt.hist(df[df.usa == 1]["ps"], bins=40, alpha=.6, label="usa", density=True); plt.hist(df[df.usa == 0]["ps"], bins=40, alpha=.6, label="não usa", density=True)
plt.xlabel("propensity score"); plt.legend(); plt.title("sobreposição"); plt.show()

Há controles em toda a faixa de scores dos tratados? Onde a comparação é frágil?

### Passo 5 — Matching 1:1 por nearest neighbor, com caliper
Para cada tratado, ache o controle com score mais próximo. Descarte pares com distância maior que `caliper = 0,02`.

In [ ]:
def parear(d, caliper=0.02):
    t, c = d[d.usa == 1], d[d.usa == 0]
    nn = NearestNeighbors(n_neighbors=1).fit(c[["ps"]])
    dist, idx = nn.kneighbors(t[["ps"]])
    ok = dist.ravel() <= caliper
    return t[ok].reset_index(drop=True), c.iloc[idx.ravel()[ok]].reset_index(drop=True)

t_p, c_p = parear(df); print(f"{len(t_p)} pares de {int(df.usa.sum())} tratados ({len(t_p) / df.usa.sum():.0%})")

### Passo 6 — O balanceamento melhorou?

In [ ]:
pareado = pd.concat([t_p, c_p])
print(pd.DataFrame({"antes": {c: smd(df, c) for c in COV}, "depois": {c: smd(pareado, c) for c in COV}}).round(3))

### Passo 7 — Estimativa (ATT)

In [ ]:
att = (t_p["colesterol"] - c_p["colesterol"]).mean()
ep = (t_p["colesterol"] - c_p["colesterol"]).std() / np.sqrt(len(t_p))
print(f"ATT por PSM: {att:.1f} ± {1.96 * ep:.1f} | ingênuo: {ingenuo:.1f} | verdade: {EFEITO_PT}")

### Passo 8 — Alternativa: IPW
Pese os tratados por `1/ps` e os controles por `1/(1−ps)` e compare as médias ponderadas (estima o ATE).

In [ ]:
w = np.where(df.usa == 1, 1 / df.ps, 1 / (1 - df.ps))
ate_ipw = np.average(df.colesterol[df.usa == 1], weights=w[df.usa == 1]) - np.average(df.colesterol[df.usa == 0], weights=w[df.usa == 0])
print(f"ATE por IPW: {ate_ipw:.1f} | verdade: {EFEITO_PT}")

### Passo 9 — O limite: unobserved confounder
Refaça tudo **sem** a variável `consc` (consciência de saúde), como se ela não existisse nos dados.

In [ ]:
d2 = df.drop(columns="ps").copy(); COV2 = ["idade", "exerc"]
d2["ps"] = LogisticRegression(max_iter=1000).fit(d2[COV2], d2["usa"]).predict_proba(d2[COV2])[:, 1]
t2, c2 = parear(d2)
att2 = (t2["colesterol"] - c2["colesterol"]).mean()
print(f"ATT sem observar `consc`: {att2:.1f} | verdade: {EFEITO_PT}")
print("balanceamento das covariáveis observadas:", {c: round(smd(pd.concat([t2, c2]), c), 3) for c in COV2})

O balanceamento parece ótimo e a estimativa está errada. Nenhum diagnóstico avisa.

## Parte B — Instrumental Variables

### Passo 10 — O cenário
A empresa quer saber se um curso de IA aumenta o salário. Funcionários mais **motivados** fazem o curso e ganham mais de qualquer jeito; motivação não é observada. Instrumento: a mesa do funcionário fica perto da sala de treinamento (atribuição de mesas é essencialmente aleatória). Efeito verdadeiro: **+500** por mês.

In [ ]:
EFEITO_CURSO = 500.0
def gerar_curso(n=20000, seed=1):
    r = np.random.default_rng(seed)
    motiv = r.normal(0, 1, n); perto = r.binomial(1, 0.5, n)
    curso = r.binomial(1, 1 / (1 + np.exp(-(-0.8 + 1.2 * motiv + 1.3 * perto))))
    salario = 6000 + 900 * motiv + EFEITO_CURSO * curso + r.normal(0, 800, n)
    return pd.DataFrame({"perto": perto, "curso": curso, "salario": salario, "motiv": motiv})

dc = gerar_curso()
ols = smf.ols("salario ~ curso", dc).fit().params["curso"]
print(f"OLS ingênuo: {ols:.0f} | verdade: {EFEITO_CURSO:.0f}")

### Passo 11 — Verificar o instrumento
Relevância: `perto` muda a taxa de adesão ao curso? Calcule a estatística F do first stage. Independência: `perto` está correlacionado com a motivação? (Só é possível checar aqui porque os dados são simulados.)

In [ ]:
primeiro = smf.ols("curso ~ perto", dc).fit()
print(f"adesão longe: {dc[dc.perto == 0].curso.mean():.2f} | perto: {dc[dc.perto == 1].curso.mean():.2f} | F do primeiro estágio: {primeiro.fvalue:.0f}")
print("correlação instrumento × motivação:", round(dc.perto.corr(dc.motiv), 3))

### Passo 12 — Estimador de Wald
`(diferença de salário entre perto e longe) / (diferença de adesão entre perto e longe)`.

In [ ]:
num = dc[dc.perto == 1].salario.mean() - dc[dc.perto == 0].salario.mean()
den = dc[dc.perto == 1].curso.mean() - dc[dc.perto == 0].curso.mean()
wald = num / den
print(f"efeito de estar perto sobre o salário: {num:.0f} | sobre a adesão: {den:.2f} | Wald: {wald:.0f} | verdade: {EFEITO_CURSO:.0f}")

### Passo 13 — 2SLS em dois passos
Guarde o tratamento previsto pelo first stage e regrida o salário nele.

In [ ]:
dc["curso_hat"] = primeiro.fittedvalues
tsls = smf.ols("salario ~ curso_hat", dc).fit().params["curso_hat"]
print(f"2SLS: {tsls:.0f} | Wald: {wald:.0f} | OLS ingênuo: {ols:.0f} | verdade: {EFEITO_CURSO:.0f}")
assert abs(tsls - wald) < 1e-6

### Passo 14 — Weak instrument
Se a proximidade quase não mudasse a adesão, o que aconteceria? Simule um weak instrument e repita a estimativa em 200 amostras.

In [ ]:
def wald_amostra(forca, seed, n=5000):
    r = np.random.default_rng(seed); motiv = r.normal(0, 1, n); perto = r.binomial(1, 0.5, n)
    curso = r.binomial(1, 1 / (1 + np.exp(-(-0.8 + 1.2 * motiv + forca * perto)))); sal = 6000 + 900 * motiv + EFEITO_CURSO * curso + r.normal(0, 800, n)
    return (sal[perto == 1].mean() - sal[perto == 0].mean()) / (curso[perto == 1].mean() - curso[perto == 0].mean())

for forca in (1.3, 0.3, 0.05):
    est = np.array([wald_amostra(forca, s) for s in range(200)])
    print(f"força {forca:4.2f} | mediana {np.median(est):8.0f} | intervalo 5%-95%: [{np.percentile(est, 5):9.0f}, {np.percentile(est, 95):9.0f}]")

**Reflexão:** com weak instrument, o denominador fica perto de zero e a estimativa explode. Por isso se exige um first stage forte antes de olhar o resultado.

## Teste Prático de Entrevista — Estimador de Wald (Instrumental Variables)
Um entrevistador pede: *"Em um encouragement design onde `z` é o incentivo sorteado (0/1), `d` é o uso real da feature (0/1) e `y` é a métrica de negócio, implemente o estimador de Wald `wald_iv(y, d, z)`. Se o denominador (first stage) for zero, lance `ValueError`."* 

In [ ]:
def wald_iv(y, d, z):
    """Estima LATE = (E[Y|Z=1]-E[Y|Z=0]) / (E[D|Z=1]-E[D|Z=0])."""
    y1 = [yv for yv, zv in zip(y, z) if zv == 1]
    y0 = [yv for yv, zv in zip(y, z) if zv == 0]
    d1 = [dv for dv, zv in zip(d, z) if zv == 1]
    d0 = [dv for dv, zv in zip(d, z) if zv == 0]
    itt_y = sum(y1) / len(y1) - sum(y0) / len(y0)
    first_stage = sum(d1) / len(d1) - sum(d0) / len(d0)
    if abs(first_stage) < 1e-12:
        raise ValueError("First stage nulo: instrumento invalido.")
    return itt_y / first_stage


late = wald_iv(
    y=[10.0, 10.0, 12.0, 12.0],
    d=[0, 0.2, 0.6, 0.6],
    z=[0, 0, 1, 1],
)
assert abs(late - 4.0) < 1e-9, f"Esperado 4.0, veio {late}"
print(f"OK! LATE estimado via Wald IV = {late:.2f}")